## MASTER ACTIVATOR TEMPLATE

In [ ]:
import sys, os
_stale = ['chromadb','gradio','sentence_transformers', 'pydantic',
          'huggingface_hub','langchain','transformers', 'albumentations']
for m in list(sys.modules.keys()):
    if any(s in m for s in _stale):
        del sys.modules[m]

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# All project paths
LIB_DIR  = "/content/drive/MyDrive/radiology_ai/libs"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
MDL_DIR  = "/content/drive/MyDrive/radiology_ai/models"
RES_DIR  = "/content/drive/MyDrive/radiology_ai/results"
RAG_DIR  = "/content/drive/MyDrive/radiology_ai/rag_papers"
CHR_DIR  = "/content/drive/MyDrive/radiology_ai/chroma_db"

for d in [LIB_DIR,DATA_DIR,MDL_DIR,RES_DIR,RAG_DIR,CHR_DIR]:
    os.makedirs(d, exist_ok=True)

if LIB_DIR in sys.path: sys.path.remove(LIB_DIR)
sys.path.insert(0, LIB_DIR)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Libs loaded | Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

## PyTorch Dataset + DataLoaders

In [ ]:
import torch, cv2, os, json
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
DISEASES = json.load(open(f"{DATA_DIR}/diseases.json"))

# Medical Image Augmentation Strategy
TRAIN_AUG = A.Compose([
    A.Resize(256, 256),
    A.RandomCrop(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.CLAHE(clip_limit=3.0, p=0.4), # Crucial for X-ray contrast
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

VAL_AUG = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.CLAHE(clip_limit=3.0, p=1.0), # Apply to validation for consistency
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# --- 2. DATASET CLASS ---
class NIHDataset(Dataset):
     def __init__(self, csv_path, transform=None, diseases=DISEASES):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.diseases = diseases

        # Verify the file_path column exists from our EDA step
        if 'file_path' not in self.df.columns:
             # Emergency fallback if Cell 4 wasn't run with absolute paths
            img_base = "/content/drive/MyDrive/radiology_ai/data/nih/images"
            self.df['file_path'] = self.df['Image Index'].apply(lambda x: os.path.join(img_base, x))

        print(f"Dataset loaded from {os.path.basename(csv_path)}: {len(self.df)} images")

     def __len__(self):
        return len(self.df)

     def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['file_path']

        # Load image (BGR to RGB)
        img = cv2.imread(img_path)
        if img is None:
            # Return a zero tensor if image is missing to prevent crash
            return torch.zeros((3, 224, 224)), torch.zeros(len(self.diseases)), row['Image Index']

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

         # Apply Albumentations
        if self.transform:
          img = self.transform(image=img)['image']

         # Convert pathology labels to tensor
        label = torch.tensor([row[d] for d in self.diseases], dtype=torch.float32)

        return img, label, row['Image Index']

# --- 3. CREATE DATALOADERS ---
print("Initializing DataLoaders...")

train_ds = NIHDataset(f"{DATA_DIR}/train.csv", transform=TRAIN_AUG)
val_ds  = NIHDataset(f"{DATA_DIR}/val.csv", transform=VAL_AUG)
test_ds = NIHDataset(f"{DATA_DIR}/test.csv", transform=VAL_AUG)

# Recommended Batch Sizes for T4 GPU
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader  = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n DataLoaders Ready!")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val:   {len(val_loader)} batches")

## Fine-Tune Model Architecture

In [ ]:
# Load pretrained TorchXRayVision DenseNet121 and prepare for fine-tuning
# Strategy:
#   1. Load TorchXRayVision weights (trained on 600K+ chest X-rays)
#   2. Freeze early layers (they already learned good chest X-ray features)
#   3. Unfreeze denseblock4 + add new classification head
#   4. Fine-tune with lower learning rate

import torch, torch.nn as nn, json, os
import torchxrayvision as xrv

# Use your project variables
DISEASES    = json.load(open(f"{DATA_DIR}/diseases.json"))
NUM_CLASSES = len(DISEASES)  # 14

class FineTunedDenseNet(nn.Module):
    """
    TorchXRayVision DenseNet121 fine-tuned for NIH 14-class detection.
    """
    def __init__(self, num_classes=14):
        super().__init__()

        # 1. Load TorchXRayVision pretrained model
        base = xrv.models.DenseNet(weights="densenet121-res224-all")

        # 2. Extract feature backbone
        self.features = base.features

        # 3. Freeze early layers (General features)
        freeze_layers = [
            'conv0', 'norm0',
            'denseblock1', 'transition1',
            'denseblock2', 'transition2',
        ]
        for name, param in self.features.named_parameters():
            if any(name.startswith(fl) for fl in freeze_layers):
                param.requires_grad = False

        # 4. New classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.BatchNorm1d(1024),
            nn.Dropout(p=0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
            # No sigmoid here; handled by BCEWithLogitsLoss
        )

        # 5. Target Layer for Notebook 4 (Grad-CAM++)
        self.gradcam_layer = self.features.denseblock4
    def forward(self, x):
        # 1. Convert 3-channel RGB to 1-channel Grayscale
        if x.shape[1] == 3:
            x = x.mean(dim=1, keepdim=True)

        # 2. CRITICAL STEP FOR TORCHXRAYVISION
        x = (x * 2048) - 1024

        # 3. Pass through backbone
        feat = self.features(x)
        feat = torch.relu(feat)

        # 4. Final classification
        return self.classifier(feat)

# Initialize
model = FineTunedDenseNet(NUM_CLASSES).to(DEVICE)

# Count parameters
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Fine-tuned DenseNet121 ready!")
print(f"    Total params      : {total:,}")
print(f"    Trainable params : {trainable:,} ({trainable/total*100:.1f}%)")
print(f"    Frozen params    : {frozen:,} ({frozen/total*100:.1f}%)")
print(f"\n Ready for Cell 3: Training Setup")

## Training Loop with Early Stopping

In [ ]:
import torch, time, json, os, numpy as np, pandas as pd
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

def compute_mean_auc(labels, probs, diseases):
    aucs = []
    for i in range(len(diseases)):
        y_true = labels[:, i]
        # Only compute AUC if both classes (0 and 1) are present in the set
        if y_true.sum() > 0 and y_true.sum() < len(y_true):
            try:
                aucs.append(roc_auc_score(y_true, probs[:, i]))
            except:
                pass
    return np.mean(aucs) if aucs else 0.0, aucs

def run_epoch(model, loader, optimizer, criterion, device, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels, _ in tqdm(loader,
                                   desc="  Train" if training else "  Val",
                                   leave=False):
            imgs, labels = imgs.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if training:
                loss.backward()
                # Gradient clipping prevents the "Exploding Gradient" problem in deep Nets
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            all_labels.append(labels.cpu().numpy())
            all_probs.append(torch.sigmoid(logits).cpu().detach().numpy())

    all_labels = np.vstack(all_labels)
    all_probs  = np.vstack(all_probs)
    mean_auc, per_auc = compute_mean_auc(all_labels, all_probs, DISEASES)
    return total_loss / len(loader), mean_auc, per_auc

def train_model(model, train_loader, val_loader, epochs=20, lr=1e-4, patience=4):
    # Calculate weights to handle class imbalance
    df_train   = pd.read_csv(f"{DATA_DIR}/train.csv")
    counts     = np.array([df_train[d].sum() for d in DISEASES], dtype=np.float32)
    # Give higher weight to rare diseases
    weights    = torch.tensor(1.0 / (counts / counts.max()), dtype=torch.float32).to(DEVICE)

    criterion  = torch.nn.BCEWithLogitsLoss(pos_weight=weights)
    optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=lr, weight_decay=1e-4)
    scheduler  = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_auc   = 0.0
    no_improve = 0
    history    = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[]}
    save_path  = f"{MDL_DIR}/finetuned_densenet121_best.pth"

    print(f"\n Fine-tuning for up to {epochs} epochs...")
    print("-" * 65)

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        tr_loss, tr_auc, _ = run_epoch(model, train_loader, optimizer, criterion, DEVICE, True)
        vl_loss, vl_auc, per_auc = run_epoch(model, val_loader, optimizer, criterion, DEVICE, False)
        scheduler.step()

        elapsed = time.time() - t0
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_auc'].append(tr_auc)
        history['val_auc'].append(vl_auc)

        flag = "BEST" if vl_auc > best_auc else ""
        print(f"Ep {epoch:02d}/{epochs} | "
              f"tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} | "
              f"val_loss={vl_loss:.4f} val_auc={vl_auc:.4f} | "
              f"{elapsed:.0f}s{flag}")

        if vl_auc > best_auc:
            best_auc   = vl_auc
            no_improve = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_auc': best_auc,
                'per_class_auc': {d: a for d, a in zip(DISEASES, per_auc)},
                'history': history,
                'diseases': DISEASES,
                'architecture': 'FineTunedDenseNet121_TorchXRayVision'
            }, save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n Early stop at epoch {epoch}")
                break

    print(f"\n Fine-tuning complete! Best val AUC: {best_auc:.4f}")
    return history, best_auc

print("Training functions ready.")

## Run Fine-Tuning + Plot Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history, best_auc = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 20,
    lr           = 1e-4,
    patience     = 4
)

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#08090c')
plots = [('train_loss','val_loss','Loss'),
         ('train_auc','val_auc','AUC'),
         (None,'val_auc','Val AUC Only')]

for ax, (tr_key, vl_key, title) in zip(axes, plots):
    ax.set_facecolor('#08090c')
    if tr_key: ax.plot(history[tr_key], label='Train', color='#4d9fff', lw=2)
    ax.plot(history[vl_key], label='Val', color='#22d3a0', lw=2)
    ax.set_title(title, color='white')
    ax.legend(facecolor='#13171f', labelcolor='white')
    ax.tick_params(colors='#5a6480')
    for sp in ax.spines.values(): sp.set_color('#1e2535')

plt.suptitle(f'Fine-Tuning Curves — Best Val AUC: {best_auc:.4f}',
             color='white', fontsize=13)
plt.tight_layout()
plt.savefig(f"{RES_DIR}/training_curves.png", dpi=150,
            facecolor='#08090c', bbox_inches='tight')
plt.show()
print(f"\n Curves saved. Best val AUC: {best_auc:.4f}")